# 01 · Neo4j connection and story-bible schema

This notebook checks that the project can reach your AuraDB instance, creates the schema, and walks through one toy example of the append-only fact history that the continuity checker relies on.

**Before running:** fill in `.env` (see the README), check the kernel (top right in VS Code) says **Continuity Editor (.venv)**, then run the cells in order, starting with Setup.

## 0. Setup

Checks that this notebook is running on the project's Python and installs any missing libraries from `requirements.txt`. If it installs something, restart the kernel (**Restart** in the toolbar) and run the notebook again.

In [ ]:
import importlib.util
import sys

print(f"Python {sys.version.split()[0]} at {sys.executable}")
if sys.version_info < (3, 11):
    raise RuntimeError("This kernel's Python is too old: click the kernel name (top right) and choose 'Continuity Editor (.venv)'.")

# Libraries this notebook uses; continuity_editor is this project's own code.
REQUIRED = ["pandas", "neo4j", "pydantic", "pydantic_settings", "continuity_editor"]
missing = [name for name in REQUIRED if importlib.util.find_spec(name) is None]
if missing:
    print("Installing missing libraries:", ", ".join(missing))
    %pip install --quiet -r ../requirements.txt -e ..
    print("Done. Restart the kernel (Restart in the toolbar), then run the notebook again.")
else:
    print("All libraries are installed.")

In [ ]:
import pandas as pd

from continuity_editor.graph import (
    Neo4jConnection,
    add_chapter,
    add_entity,
    add_fact,
    add_novel,
    apply_schema,
    describe_schema,
    reset_novel,
)
from continuity_editor.graph.models import Chapter, Character, Fact, FactCategory, Novel

conn = Neo4jConnection()
conn.verify()

## 1. Create constraints and indexes

Safe to re-run: every statement uses `IF NOT EXISTS`, so a second run reports nothing added.

In [ ]:
print(apply_schema(conn))
schema = describe_schema(conn)
pd.DataFrame(schema["constraints"])

In [ ]:
pd.DataFrame(schema["indexes"])

## 2. A toy novel: Chapter 1

Chapter 1 introduces Mira Voss: blue eyes, age 25. Each claim becomes its own `Fact` node, linked to Mira and to the chapter that stated it.

In [ ]:
NOVEL = "toy_novel"
add_novel(conn, Novel(novel_id=NOVEL, title="Toy Novel"))

ch1 = Chapter(novel_id=NOVEL, number=1, title="Arrival")
mira = Character(novel_id=NOVEL, name="Mira Voss", aliases=["Mira"])
add_chapter(conn, ch1)
add_entity(conn, mira, ch1)

add_fact(conn, Fact(
    novel_id=NOVEL, subject_id=mira.entity_id, chapter_number=1,
    attribute="eye_colour", value="blue", category=FactCategory.CHARACTERISATION,
    evidence_span="Mira looked up, her blue eyes sharp in the lamplight.",
))
add_fact(conn, Fact(
    novel_id=NOVEL, subject_id=mira.entity_id, chapter_number=1,
    attribute="age", value="25", category=FactCategory.CHARACTERISATION,
    evidence_span="At twenty-five, Mira was the youngest captain in the fleet.",
))
mira.entity_id

## 3. What retrieval sees when Chapter 3 arrives

Before Chapter 3's facts are written, pull everything established about Mira in earlier chapters. Step 3 (retrieval) builds on this query, and its output is what Step 4 (the checker) compares each new claim against.

In [ ]:
HISTORY = """
MATCH (e:Entity {entity_id: $entity_id})-[:HAS_FACT]->(f:Fact)-[:STATED_IN]->(ch:Chapter)
WHERE ch.number < $before_chapter
RETURN ch.number AS chapter, f.attribute AS attribute, f.value AS value, f.evidence_span AS evidence
ORDER BY chapter, attribute
"""
pd.DataFrame(conn.query(HISTORY, entity_id=mira.entity_id, before_chapter=3))

## 4. Chapter 3 contradicts Chapter 1

Chapter 3 says Mira's eyes are green. The new fact is **created, not overwritten**, so the full history now holds both eye colours, each with the chapter and quote that stated it. That pair is exactly what the checker will flag.

In [ ]:
ch3 = Chapter(novel_id=NOVEL, number=3, title="The Harbour")
add_chapter(conn, ch3)
add_entity(conn, mira, ch3)
add_fact(conn, Fact(
    novel_id=NOVEL, subject_id=mira.entity_id, chapter_number=3,
    attribute="eye_colour", value="green", category=FactCategory.CHARACTERISATION,
    evidence_span="Mira's green eyes narrowed at the harbourmaster.",
))

# before_chapter=4: everything up to and including Chapter 3
pd.DataFrame(conn.query(HISTORY, entity_id=mira.entity_id, before_chapter=4))

## 5. Clean up

Delete the toy novel so it doesn't mix with real data. Skip this cell if you want to explore the graph first in the Aura console (Query tab: `MATCH (n {novel_id: 'toy_novel'}) RETURN n`).

In [ ]:
print(reset_novel(conn, NOVEL))
conn.close()